In [0]:
# Ranking - row_number(),rank(), dense_rank()

data =[ ('Lisa', 'Sales', 10000, 35),
          ('Evan', 'Sales', 32000, 38),
          ('Fred', 'Engineering', 21000, 28),
          ('Alex', 'Sales', 30000, 33),
          ('Tom', 'Engineering', 23000, 33),
          ('Jane', 'Marketing', 29000, 28),
          ('Jeff', 'Marketing', 35000, 38),
          ('Paul', 'Engineering', 29000, 23),
          ('Chloe', 'Engineering', 23000, 25)]

df = spark.createDataFrame(data, ['name', 'dept', 'salary', 'age'])
df.show()

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w_spec = Window.partitionBy("dept").orderBy("salary")
df.withColumn("row_number", row_number().over(w_spec)) \
  .withColumn("rank", rank().over(w_spec)) \
  .withColumn("dense_rank", dense_rank().over(w_spec)) \
  .show()


In [0]:
#window + value or analytical
#lead -- next value in the window
#lag - prvious value in the window
#first value 
#last value (have to check this)

from pyspark.sql.window import Window
from pyspark.sql.functions import *

sch_str="txnid long,txndt string,custid long,amt float,cat string,prod string,city string,state string,spendby string"

txn_df = spark.read.csv("/Volumes/izwe48catalog/we48db/staging/txns_2025.txt",schema=sch_str)

txn_df1 = txn_df.withColumn("txndt",to_date(col("txndt"),"MM-dd-yyyy"))

w_spec = Window.partitionBy("custid").orderBy("txndt")

cust_txn_sale_df = txn_df1.withColumn("prev_amt",lag("amt").over(w_spec)) \
                          .withColumn("next_amt",lead("amt").over(w_spec)) \
                          .withColumn("first_amt",first("amt").over(w_spec)) \
                          .withColumn("last_amt",last("amt").over(w_spec))


cust_sel_df=cust_txn_sale_df.select("custid","txndt","amt","prev_amt","next_amt","first_amt","last_amt")

cust_sel_df.filter("custid=4000001").show()




In [0]:
# window + aggregation
# count,sum,avg,min,max

from pyspark.sql.window import Window

data =[ ('Lisa', 'Sales', 10000, 35),
          ('Evan', 'Sales', 32000, 38),
          ('Fred', 'Engineering', 21000, 28),
          ('Alex', 'Sales', 30000, 33),
          ('Tom', 'Engineering', 23000, 33),
          ('Jane', 'Marketing', 29000, 28),
          ('Jeff', 'Marketing', 35000, 38),
          ('Paul', 'Engineering', 29000, 23),
          ('Chloe', 'Engineering', 23000, 25)]

df = spark.createDataFrame(data, ['name', 'dept', 'salary', 'age'])

# Printing aggregation function as such
# df.groupBy("dept").agg(sum("salary"),avg("salary"),max("salary"),min("salary")).show()

# Printing aggregation function as window function
w_spec = Window.partitionBy("dept")

sal_agg_df = df.withColumn("sal_avg", avg("salary").over(w_spec)) \
               .withColumn("sal_max", max("salary").over(w_spec)) \
               .withColumn("sal_min", min("salary").over(w_spec)) \
               .withColumn("sal_sum", sum("salary").over(w_spec))

sal_agg_df.show()   

In [0]:
# Running total /Cumulative sum

from pyspark.sql.window import Window

w_spec=Window.partitionBy("custid").orderBy("txndt")

cum_sum_df = txn_df.withColumn("Cumulative_Sum",sum("amt").over(w_spec))

cum_sum_df.filter("custid=4000001").select("custid","txndt","amt","Cumulative_Sum").show()


In [0]:
from pyspark .sql.functions import col

#Group by

txn_df1=txn_df.groupBy("city","cat","prod").sum("amt")
txn_df1.filter((col("city")=="New York") & (col("cat")=="Exercise & Fitness")).show()

# Roll up - hierachical combination of group by
txn_df2=txn_df.rollup("city","cat","prod").sum("amt")
txn_df2.filter((col("city")=="New York") & (col("cat")=="Exercise & Fitness")).show()

#cube - all possible cobination 
txn_df3=txn_df.cube("city","cat","prod").sum("amt").orderBy("city")
txn_df3.filter((col("city")=="New York") & (col("cat")=="Exercise & Fitness")).show()

# pivot
display(txn_df.groupBy("city").pivot("cat").sum("amt"))
